## CÂU HỎI

### 1. Y có nên là nhị phân?
**Trả lời:** Có.
**Giải thích:**
- Mục tiêu của bài toán là xác định học viên có "Qua môn" hay không.
- Đây là bài toán phân loại nhị phân với nhãn $Y \in \{0, 1\}$:
  - $Y = 1$: Sinh viên Đạt / Qua môn
  - $Y = 0$: Sinh viên Không đạt / Rớt môn
- Việc mô hình hóa biến mục tiêu dưới dạng nhị phân cho phép mô hình ước lượng xác suất hậu nghiệm $P(Y=1|X) \in [0, 1]$, làm cơ sở linh hoạt để lựa chọn ngưỡng quyết định và đánh giá bằng các chỉ số phân loại chuyên biệt.

### 2. Có cần chuẩn hóa X?
**Trả lời:** Có.
**Giải thích:**
- Các biến đầu vào có thang đo và đơn vị khác biệt nhau:
  - Giờ học/tuần: 0.0 - 20.0 giờ.
  - Tỷ lệ chuyên cần: 30.0 - 100.0%.
  - Số bài tập đã nộp: 0 - 10 bài.
  - Điểm giữa kỳ (TX 1): 0.0 - 10.0 điểm.
- Với mô hình Logistic Regression, đặc trưng có thang đo lớn (tỷ lệ chuyên cần) sẽ lấn át hàm mất mát so với đặc trưng có thang đo nhỏ.
- Chuẩn hóa bằng `StandardScaler` đưa tất cả các đặc trưng về cùng phân phối (trung bình 0, độ lệch chuẩn 1), giúp quá trình tối ưu hội tụ nhanh, ổn định và trọng số $\beta$ phản ánh khách quan mức độ đóng góp của từng đặc trưng (Odds Ratio tính trên 1 độ lệch chuẩn).

### 3. Chọn threshold thế nào?
**Trả lời:**
- Ngưỡng mặc định là $0.5$ ($P \ge 0.5 \Rightarrow Y=1$).
- Vì mô hình dự báo xác suất sinh viên Đạt / Qua môn ($p = P(Y=1|X)$):
  - $p \ge \tau \Rightarrow$ Dự báo Đạt ($Y=1$).
  - $p < \tau \Rightarrow$ Dự báo Rớt môn / Cần can thiệp nguy cơ ($Y=0$).
- Khi điều chỉnh ngưỡng quyết định:
  - **Tăng ngưỡng $\tau$ (ví dụ lên $0.70$ hoặc $0.85$):** Mô hình trở nên thận trọng và khắt khe hơn khi kết luận Đạt. Những sinh viên có xác suất qua môn chưa đủ cao sẽ bị xếp vào nhóm nguy cơ ($Y=0$). Nhờ đó, nhà trường phát hiện được nhiều sinh viên có nguy cơ rớt hơn (Recall lớp Rớt tăng mạnh từ 75.0% tại $\tau=0.50$ lên 100.0% tại $\tau=0.85$, Balanced Accuracy đạt 97.7% trên tập test minh họa).
  - **Hạ ngưỡng $\tau$ (ví dụ xuống $0.30$):** Mô hình dễ dãi hơn khi công nhận Đạt, khiến nhiều sinh viên nguy cơ bị bỏ sót (Recall lớp Rớt giảm từ 75.0% xuống chỉ còn 33.3%, Balanced Accuracy chỉ còn 66.7%).
  - Vì vậy, với bài toán cảnh báo học tập nhằm can thiệp sớm và không bỏ sót sinh viên yếu kém, ta nên **tăng ngưỡng** phân loại $\tau$. Ngưỡng được lựa chọn bằng cross-validation trên tập train hoặc trên tập validation riêng. Tập test chỉ được sử dụng để đánh giá cuối cùng.
  - Ba ngưỡng 0.30, 0.50, 0.85 hiện chỉ dùng để minh họa thì vẫn hợp lệ, nhưng không nên gọi 0.85 là "ngưỡng tối ưu".

### 4. Metric nào là quan trọng?
**Trả lời:** **Recall lớp Rớt**, **Macro F1** và **Balanced Accuracy**.
**Giải thích:**
- Dữ liệu có sự mất cân bằng giữa hai lớp (tỷ lệ sinh viên qua môn chiếm đa số ~89.7%, trong khi rớt môn chỉ chiếm ~10.3%).
- Nếu chỉ dựa vào Accuracy thông thường (97.5%) hay F1 của riêng lớp Đạt (98.6%), người xem rất dễ ngộ nhận mô hình hoạt động hoàn hảo trong khi thực tế có thể bỏ sót sinh viên gặp khó khăn.
- Trong giáo dục, trường hợp sinh viên thực tế rớt nhưng được dự báo đạt là FP theo quy ước lớp Đạt là lớp dương. Đây cũng là một ca rớt bị bỏ sót trong nghiệp vụ cảnh báo, gây hậu quả nghiêm trọng vì sinh viên sẽ chủ quan và không nhận được sự hỗ trợ kịp thời.
- Do đó, **Recall lớp Rớt (lớp thiểu số)** là chỉ số nhạy cảm và quan trọng hàng đầu để đánh giá năng lực phát hiện sinh viên nguy cơ.
- Đồng thời, **Macro F1** (trung bình F1 của 2 lớp) và **Balanced Accuracy** (trung bình Recall của 2 lớp) là các chỉ số toàn diện nhất để đo lường hiệu quả mô hình trên tập dữ liệu mất cân bằng.

## LỰA CHỌN MÔ HÌNH 
- **Mô hình lựa chọn:** **Hồi quy Logistic (Logistic Regression)**.
- **Lý do lựa chọn:**
  1. **Đầu ra xác suất trực tiếp:** Hàm Sigmoid ánh xạ đầu ra về khoảng $[0, 1]$, trực tiếp cung cấp xác suất qua môn theo đúng yêu cầu hiển thị của Dashboard.
  2. **Tính giải thích cao:** Cho phép quan sát trọng số và chiều ảnh hưởng của từng yếu tố học tập (giờ học, chuyên cần, bài tập, điểm số) đối với kết quả qua môn thông qua hệ số $\beta$ chuẩn hóa và tỷ số Odds Ratio (OR).
  3. **Độ ổn định cao và kiểm soát Overfitting:** Với bốn thuộc tính và 600 mẫu, Logistic Regression có độ phức tạp thấp và nguy cơ quá khớp tương đối nhỏ. Khả năng khái quát vẫn cần được kiểm tra trên tập test hoặc bằng cross-validation.
  4. **Hỗ trợ điều chỉnh Threshold theo thời gian thực:** Thuận tiện trong việc thay đổi ngưỡng quyết định để quan sát ma trận nhầm lẫn và các chỉ số đo lường trên Dashboard.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, classification_report


## TẠO DỮ LIỆU
> **Lưu ý:** Tập dữ liệu gồm 600 sinh viên (538 Đạt, 62 Rớt) được mô phỏng toán học phục vụ việc minh họa trực quan thuật toán Logistic Regression, phân tích các thuộc tính học tập và nghiên cứu tác động của ngưỡng phân loại. Hiệu quả cao của mô hình (Accuracy ~97.5%) là hợp lý do nhãn được tạo trực tiếp từ các biến đầu vào, không nên diễn giải đây là hiệu quả dự kiến trên sinh viên thực tế.

In [2]:
np.random.seed(42)
n_samples = 600

gio_hoc = np.random.uniform(0, 20, n_samples)
chuyen_can = np.random.uniform(30, 100, n_samples)
so_bai_tap = np.random.randint(0, 11, n_samples)
diem_giua_ky = np.random.uniform(0.0, 10.0, n_samples)

diem_tong = (diem_giua_ky * 0.5 +
    (so_bai_tap / 10.0) * 2.0 +
    (chuyen_can / 100.0) * 2.0 +
    (gio_hoc / 20.0) * 1.0 +
    np.random.normal(0, 0.4, n_samples)
)

qua_mon = (diem_tong >= 3.0).astype(int)

df = pd.DataFrame({
    'gio_hoc_tuan': np.round(gio_hoc, 2),
    'ty_le_chuyen_can': np.round(chuyen_can, 2),
    'so_bai_tap': so_bai_tap,
    'diem_giua_ky': np.round(diem_giua_ky, 2),
    'qua_mon': qua_mon
})

df.to_csv('du_lieu_sinh_vien.csv', index=False)
print("Kich thuoc du lieu:", df.shape)
print(df.head(10))

Kich thuoc du lieu: (600, 5)
   gio_hoc_tuan  ty_le_chuyen_can  so_bai_tap  diem_giua_ky  qua_mon
0          7.49             41.83           6          6.24        1
1         19.01             49.50          10          0.89        1
2         14.64             42.39           0          7.55        1
3         11.97             36.21           5          1.28        1
4          3.12             38.44           2          8.26        1
5          3.12             62.25          10          7.82        1
6          1.16             44.44           9          7.09        1
7         17.32             55.50           6          0.36        1
8         12.02             65.24           9          3.03        1
9         14.16             78.33           1          2.63        1


### Thống kê tổng quan dữ liệu và phân phối nhãn qua môn

In [3]:
print(df.info())
print(df.describe())
print(df['qua_mon'].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gio_hoc_tuan      600 non-null    float64
 1   ty_le_chuyen_can  600 non-null    float64
 2   so_bai_tap        600 non-null    int32  
 3   diem_giua_ky      600 non-null    float64
 4   qua_mon           600 non-null    int64  
dtypes: float64(3), int32(1), int64(1)
memory usage: 21.2 KB
None
       gio_hoc_tuan  ty_le_chuyen_can  so_bai_tap  diem_giua_ky     qua_mon
count    600.000000        600.000000  600.000000    600.000000  600.000000
mean      10.060167         64.670433    4.896667      5.028450    0.896667
std        5.966734         20.367140    3.131400      2.906149    0.304648
min        0.100000         30.320000    0.000000      0.030000    0.000000
25%        4.840000         46.295000    2.000000      2.492500    1.000000
50%       10.350000         65.020000    5.00

## TIỀN XỬ LÝ VÀ HUẤN LUYỆN MÔ HÌNH
Chia dữ liệu thành 80% Train (480 sinh viên) và 20% Test (120 sinh viên) có stratify, chuẩn hóa với StandardScaler và huấn luyện Logistic Regression.

In [4]:
X = df[['gio_hoc_tuan', 'ty_le_chuyen_can', 'so_bai_tap', 'diem_giua_ky']]
y = df['qua_mon']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

y_prob_test = model.predict_proba(X_test_scaled)[:, 1]
y_pred_default = (y_prob_test >= 0.5).astype(int)

print(classification_report(y_test, y_pred_default, target_names=['Rot mon (0)', 'Qua mon (1)']))

              precision    recall  f1-score   support

 Rot mon (0)       1.00      0.75      0.86        12
 Qua mon (1)       0.97      1.00      0.99       108

    accuracy                           0.97       120
   macro avg       0.99      0.88      0.92       120
weighted avg       0.98      0.97      0.97       120



## KHẢO SÁT ẢNH HƯỞNG CỦA NGƯỠNG PHÂN LOẠI 
Đánh giá mô hình trên tập Test (N=120) tại ba ngưỡng minh họa 0.30, 0.50, 0.85 để kiểm chứng tác động tới khả năng phát hiện sinh viên nguy cơ rớt môn (lưu ý: việc chọn ngưỡng tối ưu trong triển khai thực tế cần được thực hiện bằng cross-validation trên tập train hoặc validation riêng).

In [5]:
eval_thresholds = [0.30, 0.50, 0.85]
for th in eval_thresholds:
    yp = (y_prob_test >= th).astype(int)
    cm = confusion_matrix(y_test, yp)
    tn, fp, fn, tp = cm.ravel()
    rec_fail = tn / (tn + fp)
    rec_pass = tp / (tp + fn)
    bal_acc = (rec_fail + rec_pass) / 2
    f1_fail = f1_score(y_test, yp, pos_label=0, zero_division=0)
    f1_pass = f1_score(y_test, yp, pos_label=1, zero_division=0)
    f1_macro = f1_score(y_test, yp, average='macro', zero_division=0)
    print(f"Nguong tau = {th:.2f} -> TN={tn}, FP={fp}, FN={fn}, TP={tp} | Recall Rot: {rec_fail*100:.1f}%, Recall Dat: {rec_pass*100:.1f}%, Balanced Acc: {bal_acc*100:.1f}%, F1 Rot: {f1_fail*100:.1f}%, Macro F1: {f1_macro*100:.1f}%")

Nguong tau = 0.30 -> TN=4, FP=8, FN=0, TP=108 | Recall Rot: 33.3%, Recall Dat: 100.0%, Balanced Acc: 66.7%, F1 Rot: 50.0%, Macro F1: 73.2%
Nguong tau = 0.50 -> TN=9, FP=3, FN=0, TP=108 | Recall Rot: 75.0%, Recall Dat: 100.0%, Balanced Acc: 87.5%, F1 Rot: 85.7%, Macro F1: 92.2%
Nguong tau = 0.85 -> TN=12, FP=0, FN=5, TP=103 | Recall Rot: 100.0%, Recall Dat: 95.4%, Balanced Accuracy: 97.7%, F1 Rot: 82.8%, Macro F1: 90.2%
